# CH06. 호출 캡슐화하기 - 커맨드 패턴(Command Pattern)

## 1. 개요
커맨드 패턴은 **요청(Request)을 객체로 캡슐화**하여, 호출자(Invoker)와 수신자(Receiver)를 분리(Decoupling)하는 패턴입니다.

### 비유: 식당의 주문 시스템
![img.png](img.png)
- 손님 (Client): 주문을 결정하고 웨이트리스에게 전달합.
- 주문서 (Command): 주문 내용이 적힌 객체. 주방장에게 무엇을 할지 알려주는 인터페이스 역할.
- 웨이트리스 (Invoker): 주문서를 받아서 주방장에게 전달 (주방장이 구체적으로 뭘 하는지는 몰라도 됨.)
- 주방장 (Receiver): 실제로 요리를 하는 객체.

## 2. 클래스 구성
![img_1.png](img_1.png)
![img_5.png](img_5.png)
- Client : ConcreteCommand를 생성하고 Receiver를 설정합니다.
- Invoker :	커맨드 객체를 저장하고 있으며, 적절한 시점에 execute()를 호출합니다.
- Command :	모든 커맨드 객체가 구현해야 하는 인터페이스. 보통 execute() 메서드 하나만 가집니다.
- ConcreteCommand :	실제 동작과 리시버를 연결합니다. execute()가 호출되면 리시버의 메서드를 호출합니다.
- Receiver : 실제로 일을 하는 객체 (예: 전등, 오디오, 차고 문).


In [9]:
// Receiver (수신자) : 리모컨이 제어할 실제 가전제품

class Light(val location: String) {
    fun on() = println("$location 전등이 켜졌습니다.")
    fun off() = println("$location 전등이 꺼졌습니다.")
}

class Stereo(val location: String) {
    fun on() = println("$location 오디오가 켜졌습니다.")
    fun setVolume(level: Int) = println("$location 오디오 볼륨이 $level 로 설정되었습니다.")
    fun off() = println("$location 오디오가 꺼졌습니다.")
}

In [10]:
// Command (명령 객체)
interface Command {
    fun execute()
    fun undo()
}

// 전등 켜기 명령
class LightOnCommand(private val light: Light) : Command {
    override fun execute() = light.on()
    override fun undo() = light.off()
}

// 오디오 켜기 명령
class StereoOnWithCDCommand(private val stereo: Stereo) : Command {
    override fun execute() {
        stereo.on()
        stereo.setVolume(11)
    }
    override fun undo() = stereo.off()
}

In [12]:
/* Invoker (호출자) : 명령이 무엇인지 모름. 그저 execute()를 호출할 슬롯을 가진 리모컨 */

class RemoteControl {
    private var slot: Command? = null
    private var lastCommand: Command? = null // Undo를 위한 기록

    fun setCommand(command: Command) {
        slot = command
    }

    fun buttonWasPressed() {
        slot?.execute()
        lastCommand = slot
    }

    fun undoButtonWasPressed() {
        print("Undo 실행: ")
        lastCommand?.undo()
    }
}

**널 커맨드 (Null Object Pattern)**
보통 리모컨의 버튼이 비어있으면 `if (slot != null)` 같은 체크가 필요.  하지만 아무 일도 하지 않는 `NoCommand` 객체를 넣어두면, 조건문 없이도 안전하게 실행할 수 있음.

In [15]:
class NoCommand : Command {
    override fun execute() { /* 아무것도 하지 않음 */ }
    override fun undo() { /* 아무것도 하지 않음 */ }
}

// Invoker 수정 (초기값을 NoCommand로 설정)
class RemoteControlWithNoCommand {
    private var slot: Command = NoCommand() // null 대신 NoCommand 사용

    fun setCommand(command: Command) {
        slot = command
    }

    fun buttonWasPressed() {
        slot.execute() // null 체크(?.) 없이 바로 호출 가능!
    }
}

In [14]:
// ~ 실습 ~

// 1. 리시버 생성
val livingRoomLight = Light("거실")
val kitchenStereo = Stereo("주방")

// 2. 커맨드 생성
val lightOn = LightOnCommand(livingRoomLight)
val stereoOn = StereoOnWithCDCommand(kitchenStereo)

// 3. 인보커(리모컨) 준비
val remote = RemoteControl()

// 4. 테스트: 전등 켜기
remote.setCommand(lightOn)
remote.buttonWasPressed()
remote.undoButtonWasPressed()

println("--------------------")

// 5. 테스트: 오디오 켜기
remote.setCommand(stereoOn)
remote.buttonWasPressed()
remote.undoButtonWasPressed()

거실 전등이 켜졌습니다.
Undo 실행: 거실 전등이 꺼졌습니다.
--------------------
주방 오디오가 켜졌습니다.
주방 오디오 볼륨이 11 로 설정되었습니다.
Undo 실행: 주방 오디오가 꺼졌습니다.


**매크로 커맨드(Macro Command)**
여러 개의 명령을 리스트로 묶어 버튼 하나로 동시에 실행하는 기능.
'파티 모드'나 '취침 모드' 같은 기능을 만들 때 유용.

In [ ]:
class MacroCommand(private val commands: List<Command>) : Command {
    override fun execute() {
        commands.forEach { it.execute() }
    }

    override fun undo() {
        // 취소는 실행의 역순으로 해야 안전
        commands.reversed().forEach { it.undo() }
    }
}

// 사용 예시
val partyMode = MacroCommand(listOf(lightOn, stereoOn))
remote.setCommand(partyMode)
remote.buttonWasPressed()

## 3. 장점
- DIP (의존 역전 원칙): 고수준 모듈(Invoker)이 저수준 모듈(Receiver)의 구체적인 구현에 의존하지 않게 됩니다.
- 확장성: 새로운 커맨드 클래스를 추가해도 기존 코드를 수정할 필요가 없습니다.
- 작업 취소(Undo): execute()의 반대 연산을 수행하는 undo() 메서드를 구현하면 쉽게 이전 상태로 되돌릴 수 있습니다.
- 매크로 커맨드: 여러 명령을 리스트로 담아 한 번에 실행할 수 있습니다. (예: '취침 모드' 버튼 하나로 전등 끄기 + 커튼 닫기)